# Step 2 — NLP Preprocessing Pipeline

**Project:** A Machine Learning Approach to Financial News Sentiment and Market Trend Prediction  
**Student:** Khelil Dhiaeddine  
**Supervisor:** Nesrine Lahiani  

---

## Overview

This notebook is the third step in the Step 2 NLP Preprocessing sub-pipeline, following `02_eda_finbert_pilot.ipynb` (FinBERT diagnostic on 200 raw rows) and `03_nlp_cleaning_by_source.ipynb` (source-specific noise removal). The input is `layer2_cleaned.parquet` — the unified, noise-free dataset of ~87k documents across four sources: `news`, `reddit`, `twitter_general`, and `twitter_musk`.

The goal here is to transform the cleaned raw text into a representation that both the classical baseline models (TF-IDF + LR/SVM, VADER) and FinBERT can work with. Because each source has a distinct language register — journalistic prose, social media slang, forum discussion — every preprocessing step is handled with source-aware logic rather than a single global transformation.

### Steps in this notebook

| # | Step | Output column |
|---|------|---------------|
| 2.1 | Tokenization | `tokens_raw` |
| 2.2 | Stopword removal | `tokens_filtered` |
| 2.3 | Lemmatization | `tokens_lemma` |
| 2.4 | Named Entity Recognition | `entities`, `has_tsla_entity` |
| 2.5 | Final export | `layer2_preprocessed.parquet` |

> **Important:** FinBERT (Sentiment Analysis, Step 3) operates on cleaned raw text, **not** on lemmatized tokens. Lemmatization is prepared exclusively for classical baselines. The `text` column (cleaned) is preserved untouched throughout this notebook.

---

## 0. Environment Setup

Kaggle comes with most NLP libraries pre-installed, but `spaCy`'s English model needs to be downloaded explicitly. Running this once per session is enough.

In [3]:
# Download the spaCy English model used for lemmatization and NER.
# en_core_web_sm is the small model — fast enough for 87k documents on CPU.
# If you need higher NER accuracy at the cost of speed, swap for en_core_web_trf (transformer-based).
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)

print("spaCy model ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy model ready.


## 1. Imports & Configuration

In [4]:
import os
import sys
from dotenv import load_dotenv

# Allow src/ imports without pip install -e .
sys.path.insert(0, os.path.abspath('..'))
load_dotenv()

import re
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import spacy
import nltk
from nltk.tokenize import word_tokenize, TweetTokenizer
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# Download required NLTK resources (punkt for word_tokenize, stopwords corpus)
nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)
nltk.download("stopwords",  quiet=True)
nltk.download("wordnet",    quiet=True)

# Load spaCy model — used for both lemmatization and NER
nlp = spacy.load("en_core_web_sm")

print("All imports loaded.")
print(f"spaCy version : {spacy.__version__}")
print(f"NLTK version  : {nltk.__version__}")
print(f"pandas version: {pd.__version__}")

All imports loaded.
spaCy version : 3.8.11
NLTK version  : 3.9.1
pandas version: 2.3.3


In [5]:
# ─── Path Configuration ────────────────────────────────────────────────────────
# Paths are read from environment variables so the notebook runs unchanged
# on Kaggle (set via Notebook secrets) and locally (set via .env).
# Fallback defaults point to the standard project directory layout.

INPUT_PATH  = Path(os.getenv('LAYER2_CLEANED_PATH', '../data/processed/layer2_cleaned.parquet'))
OUTPUT_PATH = Path(os.getenv('OUTPUT_PATH', '../data/processed')) / 'layer2_preprocessed.parquet'

os.makedirs(OUTPUT_PATH.parent, exist_ok=True)

# Sources present in the dataset — used for source-conditional logic throughout
TWITTER_SOURCES = {"twitter_general", "twitter_musk"}
REDDIT_SOURCES  = {"reddit"}
NEWS_SOURCES    = {"news"}

# ─── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42

print(f'Input  : {INPUT_PATH}')
print(f'Output : {OUTPUT_PATH}')


## 2. Load & Inspect the Cleaned Dataset

Before touching any text, a quick sanity check confirms the schema and row counts are exactly what we expect coming out of the Day 2 cleaning notebook.

In [6]:
df = pd.read_parquet(INPUT_PATH)

print(f"Rows loaded  : {len(df):,}")
print(f"Columns      : {list(df.columns)}")
print(f"\nSource distribution:")
print(df["source"].value_counts().to_string())
print(f"\nNull values per column:")
print(df.isnull().sum().to_string())

Rows loaded  : 87,196
Columns      : ['doc_id', 'published_at', 'source', 'text', 'ticker', 'url']

Source distribution:
source
news               43680
twitter_general    37422
reddit              3339
twitter_musk        2755

Null values per column:
doc_id            0
published_at      0
source            0
text              0
ticker            0
url             932


In [7]:
# Quick visual check — one sample per source to confirm cleaning carried through correctly
for src in df["source"].unique():
    sample = df[df["source"] == src]["text"].iloc[0]
    print(f"\n[{src}]\n{sample[:300]}")


[news]
MSN

[twitter_musk]
Congratulations Tesla &amp; SpaceX on great 2019! You rock!! Looking forward to epic 2020 ♥️🚀🛰🚘☀️

[twitter_general]
In other words, AMD has been giving Tesla preferential treatment bc it knows that most of the legacy OEMs will be gone in a few years. Very smart.

$TSLA https://t.co/XvkJhCd4VQ

[reddit]
23% of Tesla's Q1 earnings were from Bitcoin trading - and they would have missed earnings otherwise. Title says it all.

Tesla's earnings in the first quarter were $438 million.

$101 million (23%) of that came from selling Bitcoin for profit.

The other interesting part is that it would have been 


---
## 3. Tokenization

Tokenization is the process of splitting raw text into individual units (tokens) that downstream steps can operate on. The key decision here is **which tokenizer to use per source**:

- **Twitter sources** → `TweetTokenizer` (NLTK). This tokenizer was built specifically for social media text. It correctly handles cashtags (`$TSLA`), emoticons, contractions (`can't`, `won't`), and repeated characters (`loooool`). Standard `word_tokenize` would break `$TSLA` into `$` + `TSLA`, losing the cashtag as a financial signal token.

- **Reddit & News** → `word_tokenize` (NLTK / Punkt). These sources use structured prose. Standard Penn Treebank tokenization is appropriate and consistent with what spaCy expects for downstream lemmatization.

All tokens are lowercased at this stage, with one exception: **cashtags and all-caps tickers are preserved in their original form** before lowercasing — we record them as NER targets first (Step 2.4), then lowercase.

In [8]:
# ─── TweetTokenizer Configuration ─────────────────────────────────────────────
# preserve_case=False  : lowercase everything uniformly
# reduce_len=True      : normalize "looool" → "loool" (max 3 repeated chars)
# strip_handles=False  : keep @mentions as tokens — they carry the Musk Effect signal
tweet_tokenizer = TweetTokenizer(
    preserve_case=False,
    reduce_len=True,
    strip_handles=False
)

In [9]:
def tokenize(text: str, source: str) -> list[str]:
    """
    Source-aware tokenization.

    Returns a list of string tokens. The text is expected to be already
    cleaned (layer2_cleaned), so no noise removal happens here — only splitting.

    Parameters
    ----------
    text   : cleaned document text
    source : one of 'twitter_general', 'twitter_musk', 'reddit', 'news'
    """
    if not isinstance(text, str) or not text.strip():
        return []

    if source in TWITTER_SOURCES:
        # TweetTokenizer handles $TSLA, #tesla, @elonmusk correctly
        return tweet_tokenizer.tokenize(text)
    else:
        # Punkt-based word tokenizer — works well for sentence-structured text
        # Lowercase manually since word_tokenize preserves case by default
        return word_tokenize(text.lower())

In [10]:
# Apply tokenization row-by-row respecting the source column
df["tokens_raw"] = df.apply(
    lambda row: tokenize(row["text"], row["source"]),
    axis=1
)

# Sanity check — token count distribution per source
df["token_count"] = df["tokens_raw"].str.len()

print("Token count statistics per source:")
print(
    df.groupby("source")["token_count"]
    .agg(["mean", "median", "min", "max"])
    .round(1)
    .to_string()
)

Token count statistics per source:
                  mean  median  min    max
source                                    
news             597.4   325.0    0  23071
reddit           310.6   103.0    1   7498
twitter_general   33.4    29.0    3    140
twitter_musk      18.1    13.0    2     70


In [11]:
# Visual inspection — compare token output per source
for src in df["source"].unique():
    sample_row = df[df["source"] == src].iloc[0]
    print(f"\n[{src}]")
    print(f"  Text    : {sample_row['text'][:150]}")
    print(f"  Tokens  : {sample_row['tokens_raw'][:20]}")


[news]
  Text    : MSN
  Tokens  : ['msn']

[twitter_musk]
  Text    : Congratulations Tesla &amp; SpaceX on great 2019! You rock!! Looking forward to epic 2020 ♥️🚀🛰🚘☀️
  Tokens  : ['congratulations', 'tesla', '&', 'spacex', 'on', 'great', '2019', '!', 'you', 'rock', '!', '!', 'looking', 'forward', 'to', 'epic', '2020', '♥', '️', '🚀']

[twitter_general]
  Text    : In other words, AMD has been giving Tesla preferential treatment bc it knows that most of the legacy OEMs will be gone in a few years. Very smart.

$T
  Tokens  : ['in', 'other', 'words', ',', 'amd', 'has', 'been', 'giving', 'tesla', 'preferential', 'treatment', 'bc', 'it', 'knows', 'that', 'most', 'of', 'the', 'legacy', 'oems']

[reddit]
  Text    : 23% of Tesla's Q1 earnings were from Bitcoin trading - and they would have missed earnings otherwise. Title says it all.

Tesla's earnings in the firs
  Tokens  : ['23', '%', 'of', 'tesla', "'s", 'q1', 'earnings', 'were', 'from', 'bitcoin', 'trading', '-', 'and', 'they', 'would

---
## 4. Stopword Removal

Stopword removal reduces vocabulary size by discarding tokens that carry no semantic content for the downstream model. We start from the NLTK English stopword list and apply two sets of manual adjustments:

### Tokens to REMOVE from the base list (i.e., keep them in the text)
These are words that NLTK considers stopwords but are critical for **financial sentiment polarity**:
- `not`, `no`, `nor` — negations that flip polarity ("not bullish" ≠ "bullish")
- `up`, `down` — directional price language
- `more`, `less`, `very`, `most` — sentiment intensifiers
- `against`, `before`, `after` — temporal and opposition language

### Tokens to ADD to the stopword list (source-specific noise)
Words that appear frequently but carry zero financial signal:
- Twitter: `rt`, `via`, `amp`, `http`, `https`, `co`
- Reddit: `edit`, `tldr`, `update`, `removed`, `deleted`, `mod`
- News: `said`, `says`, `according`, `read`, `click`, `subscribe`

> **Note:** Stopword removal applies to all sources for the baseline pipeline. When we use FinBERT in Step 3, we feed it the full cleaned `text` column — FinBERT handles context natively and does not benefit from stopword removal.

In [12]:
# ─── Base NLTK English Stopwords ──────────────────────────────────────────────
base_stopwords = set(stopwords.words("english"))

# ─── Financial sentiment tokens to KEEP (remove from stopword list) ───────────
# These would normally be filtered out by NLTK but carry real signal here.
sentiment_critical = {
    # Negations — critical for polarity ("not good" vs "good")
    "not", "no", "nor", "never", "neither",
    # Directional price language
    "up", "down", "above", "below", "over", "under",
    # Intensifiers — affect sentiment strength
    "more", "less", "very", "most", "least", "much",
    # Opposition and temporal markers
    "against", "before", "after", "despite", "but", "however"
}

# Remove critical tokens from the base list so they pass through filtering
base_stopwords -= sentiment_critical

# ─── Source-specific noise tokens to ADD to the stopword list ─────────────────
twitter_noise = {
    "rt", "via", "amp", "http", "https", "co",
    "the",  # already in base but being explicit
    "lol", "lmao", "omg", "smh", "imo", "imho"
}

reddit_noise = {
    "edit", "tldr", "update", "mod", "removed", "deleted",
    "comment", "post", "thread", "op", "upvote", "downvote",
    "karma", "subreddit"
}

news_noise = {
    "said", "says", "according", "told", "reported",
    "read", "click", "subscribe", "newsletter", "copyright",
    "rights", "reserved", "advertisement"
}

# Combine all noise sets into one extended stopword vocabulary per source
# We build per-source sets so the noise additions are applied conditionally
STOPWORDS = {
    "twitter_general" : base_stopwords | twitter_noise,
    "twitter_musk"    : base_stopwords | twitter_noise,
    "reddit"          : base_stopwords | reddit_noise,
    "news"            : base_stopwords | news_noise,
}

print(f"Base stopwords (after keeping sentiment words): {len(base_stopwords)}")
print(f"Twitter stopword count : {len(STOPWORDS['twitter_general'])}")
print(f"Reddit stopword count  : {len(STOPWORDS['reddit'])}")
print(f"News stopword count    : {len(STOPWORDS['news'])}")

# Confirm our critical words are NOT in the final sets
for critical_word in ["not", "up", "down", "very"]:
    status = "KEPT ✓" if critical_word not in STOPWORDS["news"] else "REMOVED ✗ — fix this!"
    print(f"  '{critical_word}': {status}")

Base stopwords (after keeping sentiment words): 182
Twitter stopword count : 194
Reddit stopword count  : 196
News stopword count    : 195
  'not': KEPT ✓
  'up': KEPT ✓
  'down': KEPT ✓
  'very': KEPT ✓


In [13]:
def remove_stopwords(tokens: list[str], source: str) -> list[str]:
    """
    Filter stopwords from a token list using the source-specific stopword set.

    Also drops:
    - Pure punctuation tokens (single non-alphanumeric characters)
    - Single-character tokens that are not meaningful ($, %, etc. were already
      handled in cleaning; leftovers here are noise)

    Parameters
    ----------
    tokens : output of tokenize()
    source : document source
    """
    sw = STOPWORDS.get(source, base_stopwords)

    filtered = [
        tok for tok in tokens
        if tok not in sw                   # not a stopword
        and len(tok) > 1                   # more than one character
        and not re.fullmatch(r"[^\w]", tok) # not pure punctuation
    ]
    return filtered

In [14]:
df["tokens_filtered"] = df.apply(
    lambda row: remove_stopwords(row["tokens_raw"], row["source"]),
    axis=1
)

# Measure how many tokens were removed per source
df["tokens_filtered_count"] = df["tokens_filtered"].str.len()
df["tokens_removed"]        = df["token_count"] - df["tokens_filtered_count"]
df["pct_removed"]           = (df["tokens_removed"] / df["token_count"].clip(lower=1) * 100).round(1)

print("Stopword removal summary per source:")
summary = df.groupby("source")[["token_count", "tokens_filtered_count", "pct_removed"]].mean().round(1)
summary.columns = ["avg tokens before", "avg tokens after", "% removed"]
print(summary.to_string())

Stopword removal summary per source:
                 avg tokens before  avg tokens after  % removed
source                                                         
news                         597.4             366.7       30.5
reddit                       310.6             151.0       48.6
twitter_general               33.4              17.3       46.8
twitter_musk                  18.1              10.5       34.6


In [15]:
# Top 20 most frequent tokens after filtering — useful to spot any remaining noise
print("Top 20 tokens after stopword removal (all sources combined):")
all_filtered_tokens = [tok for tokens in df["tokens_filtered"] for tok in tokens]
top_tokens = Counter(all_filtered_tokens).most_common(20)
for rank, (token, count) in enumerate(top_tokens, 1):
    print(f"  {rank:>2}. {token:<20} {count:,}")

Top 20 tokens after stopword removal (all sources combined):
   1. tesla                130,437
   2. 's                   111,558
   3. de                   100,896
   4. ...                  76,753
   5. company              66,006
   6. ''                   65,459
   7. more                 64,833
   8. musk                 64,502
   9. ``                   61,595
  10. but                  60,588
  11. not                  53,183
  12. market               51,837
  13. new                  45,577
  14. stock                45,280
  15. year                 44,052
  16. la                   43,720
  17. tsla                 43,412
  18. up                   42,494
  19. der                  41,774
  20. also                 41,016


---
## 5. Lemmatization

Lemmatization maps each token to its dictionary base form (lemma) using morphological analysis. For example:
- `bought` → `buy`
- `falling` → `fall`  
- `batteries` → `battery`

### Why spaCy over NLTK WordNetLemmatizer?
NLTK's lemmatizer requires you to specify the part-of-speech tag manually. Without it, everything is treated as a noun: `bought` → `bought` (no change). spaCy's pipeline runs POS tagging automatically before lemmatization, so `bought` is correctly tagged as a past-tense verb and maps to `buy`.

### Important architectural note
This lemmatized output (`tokens_lemma`) is **only for the classical baseline pipeline**: TF-IDF vectorization feeding Logistic Regression, SVM, and VADER. FinBERT uses WordPiece tokenization internally and must receive the raw cleaned text — lemmatizing first would distort the subword vocabulary it was pretrained on.

In [16]:
# Tokens we do NOT want spaCy to lemmatize — financial terms that should stay
# in their original form because their morphology carries meaning.
LEMMA_PROTECT = {
    "tesla", "tsla", "musk", "elon",
    "bullish", "bearish",             # already base forms but protect from edge cases
    "ipo", "etf", "sec", "ceo", "ev",  # acronyms — spaCy might mangle these
}


def lemmatize_tokens(tokens: list[str]) -> list[str]:
    """
    Lemmatize a token list using spaCy.

    We join the tokens back into a space-separated string first so spaCy can
    run its full pipeline (POS tagger → dependency parser → lemmatizer).
    The POS context is what makes spaCy's lemmatization accurate.

    Protected financial terms (LEMMA_PROTECT) are passed through unchanged.

    Parameters
    ----------
    tokens : output of remove_stopwords()
    """
    if not tokens:
        return []

    # Re-join for spaCy — it needs a string, not a list
    text_joined = " ".join(tokens)
    doc = nlp(text_joined)

    lemmas = []
    for token in doc:
        # Skip whitespace tokens spaCy may introduce
        if token.is_space:
            continue
        tok_lower = token.text.lower()
        if tok_lower in LEMMA_PROTECT:
            lemmas.append(tok_lower)    # keep the protected term as-is
        else:
            lemmas.append(token.lemma_.lower())

    return lemmas

In [17]:
# spaCy's pipe() method is significantly faster than row-by-row apply() for large datasets.
# We batch process using nlp.pipe() on the joined token strings.

print("Running lemmatization via spaCy pipe (batch mode)...")

# Build the list of joined strings for the entire dataframe
joined_texts = df["tokens_filtered"].apply(lambda toks: " ".join(toks) if toks else "")

lemma_results = []
for doc in nlp.pipe(joined_texts, batch_size=256, n_process=1):
    lemmas = [
        (token.text.lower() if token.text.lower() in LEMMA_PROTECT else token.lemma_.lower())
        for token in doc
        if not token.is_space
    ]
    lemma_results.append(lemmas)

df["tokens_lemma"] = lemma_results

print(f"Lemmatization complete. {len(df):,} documents processed.")

Running lemmatization via spaCy pipe (batch mode)...
Lemmatization complete. 87,196 documents processed.


In [18]:
# Validate lemmatization on a few examples — specifically check that
# tense and morphological forms are being normalized correctly

test_cases = [
    "bought selling falling batteries",
    "bullish bearish tsla tesla musk",
    "investors were shorting the stock aggressively"
]

print("Lemmatization sanity check:")
for text in test_cases:
    tokens = word_tokenize(text.lower())
    lemmas = lemmatize_tokens(tokens)
    print(f"  Input  : {tokens}")
    print(f"  Output : {lemmas}")
    print()

Lemmatization sanity check:
  Input  : ['bought', 'selling', 'falling', 'batteries']
  Output : ['buy', 'sell', 'fall', 'battery']

  Input  : ['bullish', 'bearish', 'tsla', 'tesla', 'musk']
  Output : ['bullish', 'bearish', 'tsla', 'tesla', 'musk']

  Input  : ['investors', 'were', 'shorting', 'the', 'stock', 'aggressively']
  Output : ['investor', 'be', 'short', 'the', 'stock', 'aggressively']



---
## 6. Named Entity Recognition (NER)

NER identifies and classifies named entities (persons, organizations, locations, etc.) in text. For this project, NER serves two specific purposes:

1. **TSLA relevance filtering** — verify that each document actually references Tesla or TSLA, catching any documents that slipped through collection with no direct mention.
2. **Musk Effect tagging** — flag documents mentioning Elon Musk explicitly, which is one of the core thesis claims (tweets by Musk → market movement).

### Strategy per source

| Source | Primary method | Why |
|--------|---------------|-----|
| Twitter | Regex rule-based | `$TSLA` cashtags are deterministic — no need for ML model overhead |
| Reddit | spaCy ORG/PERSON + regex | Mentions are text-based ("Tesla", "Musk"), regex misses paraphrases |
| News | spaCy ORG/PERSON/GPE | Journalistic text has rich entity mentions, spaCy model performs well here |

The final output per document is:
- `entities` — a JSON-serializable list of `{text, label}` dicts for each entity found
- `has_tsla_entity` — boolean flag: True if Tesla / TSLA / Musk appears as an entity

In [19]:
# ─── Rule-based patterns (regex) ──────────────────────────────────────────────
# These fire before spaCy NER and are fast + deterministic.

# Cashtag pattern — matches $TSLA and similar ticker symbols
CASHTAG_RE = re.compile(r"\$([A-Z]{1,5})\b")

# Tesla mentions — covers full name, ticker without $, and common abbreviations
TESLA_RE = re.compile(
    r"\b(tesla|tsla|tesla\s+inc\.?|tesla\s+motors)\b",
    re.IGNORECASE
)

# Musk mentions — first name, last name, full name, and common handles
MUSK_RE = re.compile(
    r"\b(elon\s+musk|elon|musk|@elonmusk)\b",
    re.IGNORECASE
)


def extract_entities_rule_based(text: str) -> list[dict]:
    """
    Fast regex-based entity extraction.
    Returns entities as [{"text": ..., "label": ...}] dicts.
    """
    entities = []

    # Cashtags → TICKER entity
    for match in CASHTAG_RE.finditer(text):
        entities.append({"text": match.group(0).upper(), "label": "TICKER"})

    # Tesla → ORG entity (if not already captured by cashtag)
    for match in TESLA_RE.finditer(text):
        entities.append({"text": match.group(0), "label": "ORG"})

    # Musk → PERSON entity
    for match in MUSK_RE.finditer(text):
        entities.append({"text": match.group(0), "label": "PERSON"})

    return entities

In [20]:
# ─── Entity labels we care about from spaCy's predictions ─────────────────────
# spaCy's en_core_web_sm uses OntoNotes 5 label set.
# We keep only the labels relevant to financial market analysis.

RELEVANT_LABELS = {
    "ORG",    # Organizations — Tesla Inc., SEC, Fed, etc.
    "PERSON", # Named individuals — Elon Musk, analysts
    "GPE",    # Geopolitical entities — countries, cities (e.g., Texas HQ move)
    "MONEY",  # Monetary values — share price mentions
    "PERCENT",# Percentages — earnings change, market share
    "DATE",   # Date expressions — earnings dates, delivery dates
    "EVENT",  # Named events — earnings call, product launch
}


def extract_entities_spacy(text: str) -> list[dict]:
    """
    spaCy-based entity extraction, filtered to financially relevant labels.
    """
    doc = nlp(text)
    entities = [
        {"text": ent.text, "label": ent.label_}
        for ent in doc.ents
        if ent.label_ in RELEVANT_LABELS
    ]
    return entities

In [21]:
def extract_entities(text: str, source: str) -> list[dict]:
    """
    Source-aware NER dispatcher.

    Twitter uses rule-based only — tweets are short, cashtags are
    the primary entity signal, and spaCy NER on 280-char noisy text
    adds noise without much gain.

    Reddit and News combine rule-based (for TSLA/Musk reliability)
    with spaCy (for richer entity context — SEC filings, earnings,
    product launches mentioned in prose).

    Deduplication removes exact duplicate entity strings within a document.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    if source in TWITTER_SOURCES:
        # Rule-based only for short tweet text
        entities = extract_entities_rule_based(text)
    else:
        # Combine both methods — rule-based first (reliable for TSLA/Musk),
        # then spaCy for additional context entities
        rule_entities  = extract_entities_rule_based(text)
        spacy_entities = extract_entities_spacy(text)

        # Merge and deduplicate by (text, label) pair
        seen = set()
        entities = []
        for ent in rule_entities + spacy_entities:
            key = (ent["text"].lower(), ent["label"])
            if key not in seen:
                seen.add(key)
                entities.append(ent)

    return entities


def has_tsla_entity(entities: list[dict]) -> bool:
    """
    Returns True if the entity list contains any mention of Tesla or Musk.
    Used as a relevance filter flag — documents with False here are
    worth reviewing for potential removal before sentiment analysis.
    """
    tsla_keywords = {"tesla", "tsla", "$tsla", "musk", "elon musk", "elon"}
    return any(
        ent["text"].lower() in tsla_keywords
        for ent in entities
    )

In [22]:
print("Running NER extraction...")

df["entities"] = df.apply(
    lambda row: extract_entities(row["text"], row["source"]),
    axis=1
)

df["has_tsla_entity"] = df["entities"].apply(has_tsla_entity)

print("NER complete.")
print(f"\nDocuments with TSLA/Musk entity: {df['has_tsla_entity'].sum():,} / {len(df):,}")
print(f"Documents WITHOUT TSLA/Musk entity: {(~df['has_tsla_entity']).sum():,}")
print(f"\nEntity coverage per source:")
print(df.groupby("source")["has_tsla_entity"].mean().mul(100).round(1).rename("% with TSLA entity").to_string())

Running NER extraction...
NER complete.

Documents with TSLA/Musk entity: 63,706 / 87,196
Documents WITHOUT TSLA/Musk entity: 23,490

Entity coverage per source:
source
news               53.8
reddit             48.7
twitter_general    99.8
twitter_musk       44.2


In [23]:
# Entity label distribution — what types of entities are being found?
all_entities = [ent for ents in df["entities"] for ent in ents]
label_counts = Counter(ent["label"] for ent in all_entities)

print("Entity label distribution (all sources):")
for label, count in label_counts.most_common():
    print(f"  {label:<12} {count:,}")

Entity label distribution (all sources):
  ORG          556,662
  PERSON       464,971
  DATE         271,217
  GPE          164,712
  MONEY        105,410
  PERCENT      88,178
  TICKER       53,686
  EVENT        3,229


In [24]:
# Example outputs per source — manually verify the extraction quality
for src in df["source"].unique():
    # Pick a row that has entities to inspect
    sample = df[(df["source"] == src) & (df["has_tsla_entity"])].iloc[0]
    print(f"\n[{src}]")
    print(f"  Text     : {sample['text'][:200]}")
    print(f"  Entities : {sample['entities'][:8]}")


[news]
  Text     : 10 Ways That Car Dealerships Can Step Up To Sell Electric Vehicles. “Most Americans aren’t interested in electric vehicles. That’s a cold fact.” Forbes, September, 2019

J.D. Power‘s Q2 2019 Mobility 
  Entities : [{'text': 'Tesla', 'label': 'ORG'}, {'text': 'n’t', 'label': 'GPE'}, {'text': 'Forbes', 'label': 'PERSON'}, {'text': 'September, 2019', 'label': 'DATE'}, {'text': 'J.D.', 'label': 'GPE'}, {'text': 'Q2 2019', 'label': 'DATE'}, {'text': '36', 'label': 'DATE'}, {'text': 'Kristin Kolodge', 'label': 'PERSON'}]

[twitter_musk]
  Text     : Congratulations Tesla &amp; SpaceX on great 2019! You rock!! Looking forward to epic 2020 ♥️🚀🛰🚘☀️
  Entities : [{'text': 'Tesla', 'label': 'ORG'}]

[twitter_general]
  Text     : In other words, AMD has been giving Tesla preferential treatment bc it knows that most of the legacy OEMs will be gone in a few years. Very smart.

$TSLA https://t.co/XvkJhCd4VQ
  Entities : [{'text': '$TSLA', 'label': 'TICKER'}, {'text': 'Tesla', 'l

In [25]:
# Review documents flagged as having no TSLA entity.
# Some of these may be legitimately off-topic and could be excluded
# from downstream analysis. We document them here for the thesis.

no_entity_df = df[~df["has_tsla_entity"]]
print(f"Documents with no TSLA entity detected: {len(no_entity_df):,}")
print("\nSample of no-entity documents:")
for _, row in no_entity_df.sample(min(5, len(no_entity_df)), random_state=RANDOM_SEED).iterrows():
    print(f"  [{row['source']}] {row['text'][:150]}")
    print()

# NOTE: We are NOT dropping these documents here.
# The decision to exclude them is a Step 3 pre-processing decision, not Step 2.
# We document the counts as a thesis data limitation note.

Documents with no TSLA entity detected: 23,490

Sample of no-entity documents:
  [news] 

  [news] 

  [news] 2022 blev kraschen för elbilarna. Höga räntor ställde till det

Det var de höga räntorna som ställde till det för biltillverkarna, där bolagen måste p

  [news] 5月汽车类上市公司市值榜：特斯拉蒸发超800亿美元 宁德时代迈入万亿阵营

  [twitter_musk] @FiveTweetTSLA @tesla_lion @archillect Seems that way



---
## 7. Final Dataset Summary & Export

All four preprocessing steps are complete. Before saving, we do a final column audit to confirm the schema matches what Step 3 (Sentiment Analysis) expects.

In [26]:
# ─── Select and order the output columns ──────────────────────────────────────
# We keep the original Layer 2 columns intact and append the new NLP columns.
# The `text` column (cleaned raw text) is preserved for FinBERT.
# Intermediate columns (token_count, tokens_removed, pct_removed) are diagnostic
# and dropped from the final export to keep the file lean.

OUTPUT_COLUMNS = [
    # ── Layer 2 original columns ──────────────────────────────────────────────
    "doc_id",          # unique document identifier
    "published_at",    # publication timestamp
    "source",          # data source
    "text",            # cleaned raw text (input to FinBERT in Step 3)
    "ticker",          # target ticker (TSLA)
    "url",             # source URL
    # ── Step 2 new columns ────────────────────────────────────────────────────
    "tokens_raw",      # post-tokenization, pre-stopword removal
    "tokens_filtered", # post-stopword removal
    "tokens_lemma",    # lemmatized — for classical baselines only
    "entities",        # NER entity list [{text, label}]
    "has_tsla_entity", # boolean relevance flag
]

df_out = df[OUTPUT_COLUMNS].copy()

print("Final output schema:")
print(df_out.dtypes.to_string())
print(f"\nTotal rows : {len(df_out):,}")
print(f"Total cols : {len(df_out.columns)}")
print(f"\nNull check:")
print(df_out.isnull().sum().to_string())

Final output schema:
doc_id                          object
published_at       datetime64[ns, UTC]
source                          object
text                            object
ticker                          object
url                             object
tokens_raw                      object
tokens_filtered                 object
tokens_lemma                    object
entities                        object
has_tsla_entity                   bool

Total rows : 87,196
Total cols : 11

Null check:
doc_id               0
published_at         0
source               0
text                 0
ticker               0
url                932
tokens_raw           0
tokens_filtered      0
tokens_lemma         0
entities             0
has_tsla_entity      0


In [27]:
# ─── Per-source statistics for the progress report ────────────────────────────
print("=" * 60)
print("STEP 2 NLP PREPROCESSING — FINAL STATISTICS")
print("=" * 60)

for src in df_out["source"].unique():
    src_df = df[df["source"] == src]
    avg_raw      = src_df["token_count"].mean()
    avg_filtered = src_df["tokens_filtered_count"].mean()
    avg_lemma    = src_df["tokens_lemma"].str.len().mean()
    tsla_pct     = src_df["has_tsla_entity"].mean() * 100
    print(f"\n  [{src}]  ({len(src_df):,} docs)")
    print(f"    Avg tokens (raw)      : {avg_raw:.1f}")
    print(f"    Avg tokens (filtered) : {avg_filtered:.1f}")
    print(f"    Avg tokens (lemma)    : {avg_lemma:.1f}")
    print(f"    TSLA entity coverage  : {tsla_pct:.1f}%")

STEP 2 NLP PREPROCESSING — FINAL STATISTICS

  [news]  (43,680 docs)
    Avg tokens (raw)      : 597.4
    Avg tokens (filtered) : 366.7
    Avg tokens (lemma)    : 387.8
    TSLA entity coverage  : 53.8%

  [twitter_musk]  (2,755 docs)
    Avg tokens (raw)      : 18.1
    Avg tokens (filtered) : 10.5
    Avg tokens (lemma)    : 10.7
    TSLA entity coverage  : 44.2%

  [twitter_general]  (37,422 docs)
    Avg tokens (raw)      : 33.4
    Avg tokens (filtered) : 17.3
    Avg tokens (lemma)    : 18.1
    TSLA entity coverage  : 99.8%

  [reddit]  (3,339 docs)
    Avg tokens (raw)      : 310.6
    Avg tokens (filtered) : 151.0
    Avg tokens (lemma)    : 180.2
    TSLA entity coverage  : 48.7%


In [28]:
# ─── Serialize list/dict columns before saving ────────────────────────────────
# Parquet can store Python lists and dicts natively, but to ensure
# compatibility across different environments (Kaggle, local, GitHub CI),
# we serialize them to JSON strings. Downstream notebooks can deserialize
# with json.loads().

for col in ["tokens_raw", "tokens_filtered", "tokens_lemma", "entities"]:
    df_out[col] = df_out[col].apply(json.dumps)

df_out.to_parquet(OUTPUT_PATH, index=False)

print(f'Saved  : {OUTPUT_PATH}')
print(f'Rows   : {len(df_out):,}')
print(f'Cols   : {list(df_out.columns)}')
print()
print('Step 2 NLP Preprocessing — complete.')



Step 2 NLP Preprocessing — complete.


---
## 8. What's Next — Step 3 (Sentiment Analysis)

The output of this notebook (`layer2_preprocessed.parquet`) feeds directly into Step 3.

**Two parallel tracks in Step 3:**

| Track | Input column | Model |
|-------|-------------|-------|
| FinBERT (Novelty 1) | `text` (cleaned raw) | `ProsusAI/finbert` via HuggingFace |
| Classical baselines | `tokens_lemma` → TF-IDF | Logistic Regression, SVM, VADER |

Both tracks produce a sentiment label (`POSITIVE` / `NEGATIVE` / `NEUTRAL`) and probability scores per document. These are then aggregated into daily sentiment indices in Step 4 (Feature Engineering).

**Thesis limitation note (to include in the report):**  
Documents with `has_tsla_entity = False` (see Cell 30 output) will be reviewed before Step 3. Those that genuinely contain no Tesla reference will be excluded from sentiment aggregation to avoid introducing noise into the market prediction features. This count should be documented in the data quality section of the thesis.